In [1]:
import numpy as np
import copy
import pandas as pd
import torch 
import torch.nn as nn 
import os 
import re
from torch.utils.data import Dataset, DataLoader 
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from collections import Counter

In [2]:
torch.manual_seed(42)

In [3]:
ROOT_DIR = os.path.abspath(os.path.join(os.path.join(os.getcwd(), ".."), ".."))
DATA_DIR = os.path.join(ROOT_DIR, "Data")
DATASET_DIR = os.path.join(DATA_DIR, "SMS-Spam-Data")
DATASET_FILE = os.path.join(DATASET_DIR, "Dataset_5971.csv")


In [4]:
df = pd.read_csv(DATASET_FILE)
df.head()

,LABEL,TEXT,URL,EMAIL,PHONE
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No
1,ham,What's up? Do you want me to come online? If y...,No,No,No
2,ham,So u workin overtime nigpun?,No,No,No
3,ham,"Also sir, i sent you an email about how to log...",No,No,No
4,Smishing,Please Stay At Home. To encourage the notion o...,No,No,No


In [5]:
tab_mask = df["TEXT"].str.startswith("\t")
print("Rows with leading \\t artifact:", tab_mask.sum())
print(df.loc[tab_mask, "LABEL"].value_counts())

df["TEXT"] = df["TEXT"].str.lstrip("\t")
assert not df["TEXT"].str.startswith("\t").any()

Rows with leading \t artifact: 153
LABEL
Smishing    125
spam         21
Spam          7
Name: count, dtype: int64


In [6]:
df.duplicated().sum(), df.shape

(np.int64(84), (5971, 5))

In [7]:
df["CLEAN"] = df["TEXT"].str.lower().str.split().str.join(" ")
df["LABEL"] = df["LABEL"].str.strip().str.lower()
df.head()

,LABEL,TEXT,URL,EMAIL,PHONE,CLEAN
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No,your opinion about me? 1. over 2. jada 3. kusr...
1,ham,What's up? Do you want me to come online? If y...,No,No,No,what's up? do you want me to come online? if y...
2,ham,So u workin overtime nigpun?,No,No,No,so u workin overtime nigpun?
3,ham,"Also sir, i sent you an email about how to log...",No,No,No,"also sir, i sent you an email about how to log..."
4,smishing,Please Stay At Home. To encourage the notion o...,No,No,No,please stay at home. to encourage the notion o...


In [8]:
df = df.drop_duplicates(subset="CLEAN").drop(columns="CLEAN")
df.shape, df.duplicated().sum()

((5831, 5), np.int64(0))

In [9]:
URL_RE = re.compile(
    r"(https?://\S+|www\.\S+|\b[a-zA-Z0-9-]+\.(?:com|co\.uk|org\.uk|org|net|info|biz)(?:/\S*)?)",
    re.IGNORECASE,
)
PHONE_RE = re.compile(r"(\+?\d[\d\-\s]{6,}\d)")

def mask_text(text: str) -> str:
    text = URL_RE.sub("<URL>", text)
    text = PHONE_RE.sub("<PHONE>", text)
    return text

df["TEXT_MASKED"] = df["TEXT"].apply(mask_text)

n_url = (df["TEXT"] != df["TEXT"].apply(lambda t: URL_RE.sub("<URL>", t))).sum()
n_phone = (df["TEXT"] != df["TEXT"].apply(lambda t: PHONE_RE.sub("<PHONE>", t))).sum()
print(f"Rows with a <URL> substitution:   {n_url}")
print(f"Rows with a <PHONE> substitution: {n_phone}")

df.loc[df["TEXT"] != df["TEXT_MASKED"], ["TEXT", "TEXT_MASKED"]].head(5)

Rows with a <URL> substitution:   220
Rows with a <PHONE> substitution: 625


,TEXT,TEXT_MASKED
5,BankOfAmerica Alert 137943. Please follow http...,BankOfAmerica Alert 137943. Please follow <URL...
23,lyricalladie(21/F) is inviting you to be her f...,lyricalladie(21/F) is inviting you to be her f...
26,UR awarded a City Break and could WIN a £200 S...,UR awarded a City Break and could WIN a £200 S...
27,Hello from Orange. For 1 month's free access t...,Hello from Orange. For 1 month's free access t...
31,New Offer! Save upto 40\% electricity bill wit...,New Offer! Save upto 40\% electricity bill wit...


In [10]:
X = df["TEXT_MASKED"]
y = df["LABEL"]

In [11]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, stratify=y_temp, random_state=42)


In [12]:
encoder = LabelEncoder()

y_train_encoded = encoder.fit_transform(y_train).tolist()
y_val_encoded = encoder.transform(y_val).tolist()
y_test_encoded = encoder.transform(y_test).tolist()

print(len(set(y_train_encoded)), len(set(y_test_encoded)), len(set(y_val_encoded)))

type(y_train_encoded)

3 3 3


list

# Tokenization for X_temp, X_train, X_val
Simple tokenization by splitting between whitespaces

In [13]:
X_train_tokenized = X_train.str.split()
X_val_tokenized = X_val.str.split()
X_test_tokenized = X_test.str.split()

In [14]:
print(X_train_tokenized.iloc[0])

['i', 'want', 'to', 'grasp', 'your', 'pretty', 'booty', ':)']


# Vocabulary building

In [15]:
# Built only from the training corpus tokens, unseen on test and validation tokens 
token_frequency_map = Counter(X_train_tokenized.explode()) # .explode().value_counts() is also an alternate option
print(token_frequency_map)

Counter({'to': 1668, 'you': 1223, 'I': 1065, 'a': 1031, 'the': 898, 'and': 622, 'in': 592, 'is': 587, 'i': 544, 'for': 520, 'u': 501, 'your': 469, 'of': 469, 'my': 458, 'me': 413, 'on': 397, 'have': 373, '<PHONE>': 347, '2': 330, 'that': 330, 'are': 310, 'call': 306, 'or': 289, 'at': 288, 'it': 283, 'with': 278, 'be': 274, 'not': 259, 'will': 256, 'can': 235, 'get': 226, 'ur': 216, 'U': 211, 'but': 209, "I'm": 209, 'from': 202, 'You': 202, 'so': 199, 'just': 187, 'this': 185, '&lt;#&gt;': 181, '4': 180, 'if': 179, 'do': 179, 'go': 178, 'up': 174, 'all': 173, '.': 169, 'when': 164, 'like': 161, 'know': 160, 'we': 154, 'out': 148, 'am': 146, 'got': 142, 'now': 141, 'come': 141, 'Call': 140, 'was': 136, '<URL>': 134, '?': 130, 'been': 129, 'about': 126, 'by': 122, 'want': 121, 'what': 121, 'has': 116, 'our': 116, 'time': 112, 'only': 112, 'Your': 111, 'then': 109, '...': 109, 'as': 106, 'need': 105, 'going': 104, 'If': 103, 'We': 102, 'one': 101, 'still': 100, 'its': 100, 'How': 98, 'But'

In [16]:
filtered_tokens = [token for token,count in token_frequency_map.most_common() if count >= 2]
index_to_token = ["PAD", "UNK"] + filtered_tokens
token_to_index = {token: index for index, token in enumerate(index_to_token)}

# Creating embeddings 

In [17]:
def create_embedding(pd_series):
    data = []
    for i in range(len(pd_series)):
        current_row = pd_series.iloc[i]
        embedding = [token_to_index.get(current_row[j], 1) for j in range(len(current_row))]
        data.append(embedding)
    return data

X_train_embedding = create_embedding(X_train_tokenized)
X_test_embedding = create_embedding(X_test_tokenized)
X_val_embedding = create_embedding(X_val_tokenized)




In [18]:
train_lengths = X_train_tokenized.apply(len)          # one integer per message
print(train_lengths.describe())          # mean, median, max
max_len = int(np.percentile(train_lengths, 95))
print("Candidate: ", max_len) 

count    4197.000000
mean       15.692399
std        10.586629
min         1.000000
25%         7.000000
50%        13.000000
75%        23.000000
max       125.000000
Name: TEXT_MASKED, dtype: float64
Candidate:  32


In [19]:
class CustomDataset(Dataset):
    def __init__(self, X, y, max_len):
        self.X = X
        self.y = y
        self.max_len = max_len

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx][:self.max_len], self.y[idx]
    

In [20]:
train_dataset = CustomDataset(X_train_embedding, y_train_encoded, max_len)
val_dataset = CustomDataset(X_val_embedding, y_val_encoded, max_len)
test_dataset = CustomDataset(X_test_embedding, y_test_encoded, max_len)


In [21]:
def collate_fn(batch):
    ids, labels = zip(*batch)                                   # unpack the (ids, label) pairs
    lengths = torch.tensor([len(x) for x in ids], dtype=torch.long)
    padded = pad_sequence(
        [torch.tensor(x, dtype=torch.long) for x in ids],
        batch_first=True,                                       # shape (batch, longest)
        padding_value=0,                                        # <PAD> index
    )
    labels = torch.tensor(labels, dtype=torch.long)
    return padded, lengths, labels

In [22]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [23]:
padded, lengths, labels = next(iter(train_loader))
print(padded.shape, padded.dtype, lengths[:5], labels.dtype)

torch.Size([32, 32]) torch.int64 tensor([12, 24, 24,  6,  3]) torch.int64


In [24]:
class CNNModel(nn.Module):
    def __init__(self, vocab_size, num_labels=3, embedding_dim=64, num_filters=100, kernel_sizes=(2,3,4), dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(embedding_dim, num_filters, k) for k in kernel_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters*len(kernel_sizes), num_labels)
        self.min_len = max(kernel_sizes)

    def forward(self, padded):
        if padded.size(1) < self.min_len:
            padded = nn.functional.pad(padded, (0, self.min-padded.size(1)), value=0)
        emb = self.embedding(padded).transpose(1,2)                 # (B, L, D)         e.g. (32, 30, 64)
        pooled = [nn.functional.relu(conv(emb)).max(dim=2).values for conv in self.convs]
        out = torch.cat(pooled, dim=1)
        return self.fc(self.dropout(out))

In [25]:
model = CNNModel(len(index_to_token))

In [26]:
counts = torch.bincount(torch.tensor(y_train_encoded), minlength=3).float()
class_weights = counts.sum() / (3 * counts)
loss_fn = nn.CrossEntropyLoss(weight=class_weights)

In [27]:
epochs = 25
learning_rate = 1e-3
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [28]:

best_f1, best_state = 0, None

for epoch in range(epochs):
    total_loss = 0
    model.train()
    for X_batch, _, y_batch in train_loader:
        y_pred = model(X_batch)

        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    model.eval()
    preds, trues = [], []
    with torch.inference_mode():
        correct = 0
        total = 0
        for X_batch, _, y_batch in val_loader:
            y_pred = model(X_batch)
            preds.extend(y_pred.argmax(dim=1).tolist())
            trues.extend(y_batch.tolist())
        
    val_f1 = f1_score(trues, preds, average="macro")
    if val_f1 > best_f1:
        best_f1 = val_f1
        cnn_best_state = copy.deepcopy(model.state_dict())

    scheduler.step()

    print(f"Epoch {epoch+1}/{epochs}  loss {avg_loss:.4f}  val macro-F1 {val_f1:.4f}  learning rate {optimizer.param_groups[0]["lr"]}")

Epoch 1/25  loss 0.9410  val macro-F1 0.6965  learning rate 0.000996057350657239


Epoch 2/25  loss 0.5919  val macro-F1 0.7705  learning rate 0.0009842915805643156


Epoch 3/25  loss 0.4059  val macro-F1 0.7629  learning rate 0.0009648882429441258


Epoch 4/25  loss 0.3139  val macro-F1 0.7905  learning rate 0.0009381533400219318


Epoch 5/25  loss 0.2422  val macro-F1 0.8680  learning rate 0.0009045084971874737


...

Epoch 21/25  loss 0.0401  val macro-F1 0.8733  learning rate 6.184665997806823e-05


Epoch 22/25  loss 0.0534  val macro-F1 0.8744  learning rate 3.511175705587433e-05


Epoch 23/25  loss 0.0552  val macro-F1 0.8744  learning rate 1.570841943568452e-05


Epoch 24/25  loss 0.0442  val macro-F1 0.8744  learning rate 3.942649342761118e-06


Epoch 25/25  loss 0.0454  val macro-F1 0.8744  learning rate 0.0


In [29]:

model.load_state_dict(cnn_best_state)   # best epoch's weights, not the last epoch's
model.eval()

preds, trues = [], []
with torch.inference_mode():
    for X_batch, _, y_batch in test_loader:
        logits = model(X_batch)
        preds.extend(logits.argmax(dim=1).tolist())
        trues.extend(y_batch.tolist())

print(classification_report(trues, preds, target_names=encoder.classes_, digits=3))
print(confusion_matrix(trues, preds))

              precision    recall  f1-score   support

         ham      0.986     0.995     0.990       967
    smishing      0.919     0.835     0.875       109
        spam      0.783     0.791     0.787        91

    accuracy                          0.964      1167
   macro avg      0.896     0.874     0.884      1167
weighted avg      0.964     0.964     0.964      1167

[[962   1   4]
 [  2  91  16]
 [ 12   7  72]]
